<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Application-Deployment/blob/main/20261030/1030_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io
import os
import zipfile
import pandas as pd
import requests

# 1. GitHub Raw 다운로드 URL 설정
# (웹 페이지 URL이 아닌 raw 파일 다운로드 경로여야 합니다)
url = "https://raw.githubusercontent.com/ancestor9/2026_Fall_Application-Deployment/main/20261023/ecommerce_tables.zip"

# 2. 파일 다운로드 및 메모리 상에서 압축 해제
response = requests.get(url)
if response.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall("./ecommerce_data")  # 현재 디렉토리의 ecommerce_data 폴더에 압축 해제
        print("✅ 압축 해제 완료:", z.namelist())
else:
    print("❌ 다운로드 실패:", response.status_code)

# 3. 압축 해제된 파일(예: CSV 또는 Excel) 읽기 예시
# 폴더 내에 포함된 파일명에 맞게 수정하여 사용하세요.
# df = pd.read_csv("./ecommerce_data/테이블명.csv")

✅ 압축 해제 완료: ['customers.csv', 'products.csv', 'stores.csv', 'orders.csv', 'order_items.csv']


In [ ]:
def summarize_df(df: pd.DataFrame) -> pd.DataFrame:
    # 데이터프레임의 각 컬럼별 데이터 타입, 결측치 수/비율, 대표값 5개를 요약한 DataFrame을 반환합니다.
    summary_data = []
    for col in df.columns:
        dtype = df[col].dtype
        n_unique = df[col].nunique()
        null_cnt = df[col].isnull().sum()
        null_ratio = round((null_cnt / len(df)) * 100, 2)
        unique_vals = df[col].dropna().unique()
        sample_vals = list(unique_vals[:7])

        summary_data.append({
            'Column': col,
            'N_Unique': n_unique,
            'Dtype': dtype,
            'Null_Count': null_cnt,
            'Null_Ratio(%)': null_ratio,
            'Sample_Values_5': sample_vals
        })

    return pd.DataFrame(summary_data)


In [ ]:

# Install the xlsxwriter library
!pip install xlsxwriter

In [ ]:
import os
import pandas as pd

# 1. 압축 해제된 폴더 내의 CSV 파일들을 불러와 dfs 딕셔너리 생성
data_dir = './ecommerce_data'
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]

dfs = {}
for file in csv_files:
    table_name = os.path.splitext(file)[0]
    file_path = os.path.join(data_dir, file)
    dfs[table_name] = pd.read_csv(file_path)
    print(f"Loaded {table_name} table (Shape: {dfs[table_name].shape})")

output_excel_path = 'all_tables_summary.xlsx'

# 2. Excel writer를 통해 시트별로 요약 데이터 작성
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:
    for table_name, df in dfs.items():
        print(f"Summarizing {table_name}...")
        summary_df = summarize_df(df)
        # 각 요약 DataFrame을 개별 시트에 기록
        summary_df.to_excel(writer, sheet_name=table_name, index=False)
        print(f"Summary for {table_name} written to sheet '{table_name}'.")

print(f"\n🎉 All table summaries saved to '{output_excel_path}'")

Loaded order_items table (Shape: (1200, 4))
Loaded products table (Shape: (30, 4))
Loaded customers table (Shape: (100, 8))
Loaded stores table (Shape: (10, 4))
Loaded orders table (Shape: (500, 4))
Summarizing order_items...
Summary for order_items written to sheet 'order_items'.
Summarizing products...
Summary for products written to sheet 'products'.
Summarizing customers...
Summary for customers written to sheet 'customers'.
Summarizing stores...
Summary for stores written to sheet 'stores'.
Summarizing orders...
Summary for orders written to sheet 'orders'.

🎉 All table summaries saved to 'all_tables_summary.xlsx'


In [ ]:
from google.colab import files

# Download the generated Excel file
files.download(output_excel_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests

# 1. GitHub Raw 다운로드 URL 설정
# 웹 브라우저 URL 주소(blob/main)를 raw.githubusercontent.com 경로로 변경합니다.
url = "https://raw.githubusercontent.com/ancestor9/2026_Fall_Application-Deployment/main/20261030/adventure_sales.db"

# 2. sqlite3 데이터베이스 파일 다운로드
response = requests.get(url)

if response.status_code == 200:
    with open("adventure_sales.db", "wb") as f:
        f.write(response.content)
    print("✅ adventure_sales.db 다운로드 완료!")
else:
    print(f"❌ 다운로드 실패 (상태 코드: {response.status_code})")

✅ adventure_sales.db 다운로드 완료!


In [ ]:
import sqlite3
import pandas as pd

# 1. 데이터베이스 연결
conn = sqlite3.connect("adventure_sales.db")

# 2. 데이터베이스 내의 모든 테이블 목록 조회 쿼리
query = "SELECT name FROM sqlite_master WHERE type='table';"

# 3. 쿼리 실행 및 결과 조회
tables = pd.read_sql_query(query, conn)
print("=== 데이터베이스 내 테이블 목록 ===")
print(tables)

# 4. 연결 종료
conn.close()

=== 데이터베이스 내 테이블 목록 ===
                   name
0      Sales_Order_data
1  Sales_Territory_data
2            Sales_data
3         Reseller_data
4             Date_data
5          Product_data
6         Customer_data


In [ ]:
import sqlite3
import pandas as pd

# 1. 데이터베이스 연결
conn = sqlite3.connect("adventure_sales.db")

# 2. 전체 테이블 목록 가져오기
tables_df = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
table_names = tables_df['name'].tolist()

# 3. 각 테이블을 데이터프레임으로 변환하여 딕셔너리에 저장
dfs_adventure = {}
for table_name in table_names:
    query = f"SELECT * FROM {table_name}"
    dfs_adventure[table_name] = pd.read_sql_query(query, conn)
    print(f"✅ 테이블 '{table_name}' -> 데이터프레임 변환 완료 (크기: {dfs_adventure[table_name].shape})")

# 4. 연결 종료
conn.close()

# 데이터프레임 사용 예시:
# dfs_adventure['Sales_Order_data'].head() 형태로 원하는 테이블의 데이터를 확인할 수 있습니다.

✅ 테이블 'Sales_Order_data' -> 데이터프레임 변환 완료 (크기: (121253, 4))
✅ 테이블 'Sales_Territory_data' -> 데이터프레임 변환 완료 (크기: (11, 4))
✅ 테이블 'Sales_data' -> 데이터프레임 변환 완료 (크기: (121253, 15))
✅ 테이블 'Reseller_data' -> 데이터프레임 변환 완료 (크기: (701, 8))
✅ 테이블 'Date_data' -> 데이터프레임 변환 완료 (크기: (1461, 7))
✅ 테이블 'Product_data' -> 데이터프레임 변환 완료 (크기: (397, 9))
✅ 테이블 'Customer_data' -> 데이터프레임 변환 완료 (크기: (18484, 7))


In [ ]:
def summarize_df(df: pd.DataFrame) -> pd.DataFrame:
    # 데이터프레임의 각 컬럼별 데이터 타입, 결측치 수/비율, 대표값 5개를 요약한 DataFrame을 반환합니다.
    summary_data = []
    for col in df.columns:
        dtype = df[col].dtype
        n_unique = df[col].nunique()
        null_cnt = df[col].isnull().sum()
        null_ratio = round((null_cnt / len(df)) * 100, 2)
        unique_vals = df[col].dropna().unique()
        sample_vals = list(unique_vals[:7])

        summary_data.append({
            'Column': col,
            'N_Unique': n_unique,
            'Dtype': dtype,
            'Null_Count': null_cnt,
            'Null_Ratio(%)': null_ratio,
            'Sample_Values_5': sample_vals
        })

    return pd.DataFrame(summary_data)

In [ ]:
dfs_adventure.keys()

dict_keys(['Sales_Order_data', 'Sales_Territory_data', 'Sales_data', 'Reseller_data', 'Date_data', 'Product_data', 'Customer_data'])

In [ ]:
for k in dfs_adventure.keys():
    print(k)
    display(summarize_df(dfs_adventure[k]))
    print("*"*100)

Sales_Order_data


,Column,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values_5
0,Channel,2,object,0,0.0,"[Reseller, Internet]"
1,SalesOrderLineKey,121253,int64,0,0.0,"[43659001, 43659002, 43659003, 43659004, 43659..."
2,Sales Order,31455,object,0,0.0,"[SO43659, SO43660, SO43661, SO43662, SO43663, ..."
3,Sales Order Line,121253,object,0,0.0,"[SO43659 - 1, SO43659 - 2, SO43659 - 3, SO4365..."


****************************************************************************************************
Sales_Territory_data


,Column,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values_5
0,SalesTerritoryKey,11,int64,0,0.0,"[1, 2, 3, 4, 5, 6, 7]"
1,Region,11,object,0,0.0,"[Northwest, Northeast, Central, Southwest, Sou..."
2,Country,7,object,0,0.0,"[United States, Canada, France, Germany, Austr..."
3,Group,4,object,0,0.0,"[North America, Europe, Pacific, Corporate HQ]"


****************************************************************************************************
Sales_data


...

****************************************************************************************************
Product_data


,Column,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values_5
0,ProductKey,397,int64,0,0.00,"[210, 211, 212, 213, 214, 215, 216]"
1,SKU,295,object,0,0.00,"[FR-R92B-58, FR-R92R-58, HL-U509-R, HL-U509, S..."
2,Product,295,object,0,0.00,"[HL Road Frame - Black, 58, HL Road Frame - Re..."
3,Standard Cost,134,float64,0,0.00,"[868.6342, 12.0278, 13.8782, 13.0863, 3.3963, ..."
4,Color,9,object,56,14.11,"[Black, Red, White, Blue, Multi, Silver, Yellow]"
5,List Price,120,float64,0,0.00,"[1431.5, 33.6442, 34.99, 9.5, 8.6442, 8.99, 48..."
6,Model,119,object,0,0.00,"[HL Road Frame, Sport-100, Mountain Bike Socks..."
7,Subcategory,37,object,0,0.00,"[Road Frames, Helmets, Socks, Caps, Jerseys, M..."
8,Category,4,object,0,0.00,"[Components, Accessories, Clothing, Bikes]"


****************************************************************************************************
Customer_data


,Column,N_Unique,Dtype,Null_Count,Null_Ratio(%),Sample_Values_5
0,CustomerKey,18484,float64,0,0.0,"[11000.0, 11001.0, 11002.0, 11003.0, 11004.0, ..."
1,Customer ID,18484,object,0,0.0,"[AW00011000, AW00011001, AW00011002, AW0001100..."
2,Customer,18400,object,0,0.0,"[Jon Yang, Eugene Huang, Ruben Torres, Christy..."
3,City,269,object,0,0.0,"[Rockhampton, Seaford, Hobart, North Ryde, Wol..."
4,State-Province,53,object,0,0.0,"[Queensland, Victoria, Tasmania, New South Wal..."
5,Country-Region,6,object,0,0.0,"[Australia, United States, Canada, Germany, Un..."
6,Postal Code,323,object,0,0.0,"[4700, 3198, 7001, 2113, 2500, 4169, 2036]"


****************************************************************************************************


In [ ]:
# @title **문제 1. 카테고리(`Category`)가 'Bikes'인 상품의 총 주문 수량(`Order Quantity`의 합계)**

# 1. 필요한 테이블 가져오기
product_df = dfs_adventure['Product_data']
sales_df = dfs_adventure['Sales_data']

# 2. ProductKey 기준 Merge
merged_df = pd.merge(sales_df, product_df, on='ProductKey', how='inner')

# 3. Category가 'Bikes'인 데이터 필터링 및 Order Quantity 합계 계산
bikes_sales = merged_df[merged_df['Category'] == 'Bikes']
total_order_quantity = bikes_sales['Order Quantity'].sum()

print(f"Category가 'Bikes'인 상품들의 총 주문 수량: {total_order_quantity:,}개")

Category가 'Bikes'인 상품들의 총 주문 수량: 90,220개


In [ ]:
# @title **문제2. 미국에서 총 매출액이 가장 높은  리셀러(`Business Type`)와 총 매출액을 구하라**
# 1. 필요한 테이블 가져오기
sales_df = dfs_adventure['Sales_data']
reseller_df = dfs_adventure['Reseller_data']
territory_df = dfs_adventure['Sales_Territory_data']

# 2. 데이터 병합 (Sales + Reseller + Territory)
# Sales_data와 Reseller_data 연결 (ResellerKey 기준)
merged_reseller = pd.merge(sales_df, reseller_df, on='ResellerKey', how='inner')
# Sales_Territory_data 연결 (SalesTerritoryKey 기준)
final_df = pd.merge(merged_reseller, territory_df, on='SalesTerritoryKey', how='inner')

# 3. 미국(United States) 데이터 필터링
us_sales = final_df[final_df['Country'] == 'United States']

# 4. 리셀러 업태(Business Type)별 총 매출액(Sales Amount) 계산
business_type_sales = us_sales.groupby('Business Type')['Sales Amount'].sum().reset_index()

# 5. 매출액 기준 내림차순 정렬
business_type_sales = business_type_sales.sort_values(by='Sales Amount', ascending=False)

print("=== 미국 내 리셀러 업태별 총 매출액 ===")
for idx, row in business_type_sales.iterrows():
    print(f"{row['Business Type']}: ${row['Sales Amount']:,.2f}")

# 가장 높은 리셀러 추출
top_business = business_type_sales.iloc[0]
print(f"\n🏆 미국에서 총 매출액이 가장 높은 리셀러 업태: {top_business['Business Type']} (${top_business['Sales Amount']:,.2f})")


=== 미국 내 리셀러 업태별 총 매출액 ===
Value Added Reseller: $25,945,862.92
Warehouse: $23,362,765.05
Specialty Bike Shop: $4,299,173.24

🏆 미국에서 총 매출액이 가장 높은 리셀러 업태: Value Added Reseller ($25,945,862.92)


In [ ]:
# @title **문제3. 2020년도에 단일 주문 매출(`Sales Amount`)이 2,200 달러 이상인 고객(`CustomerKey`)은 몇 명(중복 없는(Unique))**

# 1. 필요한 테이블 가져오기
sales_df = dfs_adventure['Sales_data']
date_df = dfs_adventure['Date_data']

# 2. DateKey를 기준으로 Sales_data와 Date_data 병합
merged_sales_date = pd.merge(sales_df, date_df, left_on='OrderDateKey', right_on='DateKey', how='inner')

# 3. 'Date' 컬럼을 datetime 형식으로 변환하여 연도 추출
merged_sales_date['Parsed_Date'] = pd.to_datetime(merged_sales_date['Date'])
merged_sales_date['Year'] = merged_sales_date['Parsed_Date'].dt.year

# 4. 2020년도 데이터 필터링 (단일 주문 매출이 $3,000 이상이며 고유 고객이어야 하므로 결측치 제거)
sales_2020_3000 = merged_sales_date[
    (merged_sales_date['Year'] == 2020) &
    (merged_sales_date['Sales Amount'] >= 2200) &
    (merged_sales_date['CustomerKey'].notnull())
]

# 5. 중복 없는 고유 고객 수(Unique CustomerKey) 계산
unique_customers_count = sales_2020_3000['CustomerKey'].nunique()

print(f"2020년도에 단일 주문 매출이 $2200 이상인 고유 고객 수: {unique_customers_count}명")

2020년도에 단일 주문 매출이 $2200 이상인 고유 고객 수: 2108명


In [ ]:
# @title **문제4. 2017년과 2018년도 2년동안 매출 Top 1 상품(ProductKey)의 매출액(`Sales Amount` 합계)이 가장 높은 상품과 매출액은?**

In [ ]:
# 1. 필요한 테이블 가져오기
sales_df = dfs_adventure['Sales_data']
date_df = dfs_adventure['Date_data']

# 2. DateKey를 기준으로 Sales_data와 Date_data 병합
merged_sales_date = pd.merge(sales_df, date_df, left_on='OrderDateKey', right_on='DateKey', how='inner')

# 3. 'Date' 컬럼을 datetime 형식으로 변환하여 연도 추출
merged_sales_date['Parsed_Date'] = pd.to_datetime(merged_sales_date['Date'])
merged_sales_date['Year'] = merged_sales_date['Parsed_Date'].dt.year

# 데이터셋 내에 존재하는 연도 리스트를 출력하여 확인합니다.
available_years = merged_sales_date['Year'].unique()
print(f"데이터셋에 존재하는 주문 연도 목록: {sorted(available_years)}")

# 4. 2021년과 2022년 데이터 필터링 (존재하는 연도에 한해서 필터링)
target_years = [year for year in [2017, 2018] if year in available_years]

if not target_years:
    # 만약 2021, 2022년 데이터가 아예 없다면 전체 기간 중 Top 1을 구하거나 에러 메시지 출력
    print("⚠️ 데이터셋에 2021년 및 2022년 데이터가 존재하지 않습니다. 대신 전체 기간의 Top 1 상품을 조회합니다.")
    sales_filtered = merged_sales_date
else:
    sales_filtered = merged_sales_date[merged_sales_date['Year'].isin(target_years)]

# 5. 상품(ProductKey)별 매출액 합계 계산
product_sales = sales_filtered.groupby('ProductKey')['Sales Amount'].sum().reset_index()

if not product_sales.empty:
    # 6. 매출액 기준 내림차순 정렬 및 Top 1 상품 추출
    top_product = product_sales.sort_values(by='Sales Amount', ascending=False).iloc[0]

    # 7. 상품 상세 정보 조회를 위해 Product_data 가져오기
    product_info_df = dfs_adventure['Product_data']
    top_product_detail = product_info_df[product_info_df['ProductKey'] == top_product['ProductKey']].iloc[0]

    print(f"\n=== {', '.join(map(str, target_years)) if target_years else '전체'} 기간 매출 Top 1 상품 결과 ===")
    print(f"🔹 상품 키 (ProductKey): {int(top_product['ProductKey'])}")
    print(f"🔹 상품명 (Product Name): {top_product_detail['Product']}")
    print(f"🔹 모델명 (Model Name): {top_product_detail['Model']}")
    print(f"🔹 총 매출액 (Sales Amount): ${top_product['Sales Amount']:,.2f}")
else:
    print("❌ 해당 기간의 매출 데이터가 존재하지 않습니다.")

데이터셋에 존재하는 주문 연도 목록: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020)]

=== 2017, 2018 기간 매출 Top 1 상품 결과 ===
🔹 상품 키 (ProductKey): 314
🔹 상품명 (Product Name): Road-150 Red, 56
🔹 모델명 (Model Name): Road-150
🔹 총 매출액 (Sales Amount): $1,847,818.63
